# Post-training evolution: from Chatbot to Agent


> Earlier lectures each treated one family of methods: lecture 06 gave the RL algorithm engines (STaR, GRPO, DAPO), lecture 07 showed the form of Agent self-improvement, and lecture 08 turned search and deep research into a concrete Agent. They sit in different chapters, each answering a local question.
>
> This lecture gathers them onto a timeline. Pretraining only gave the model the ability to **continue text**. We follow one model through its life and watch how it became a compliant assistant, then an Agent that uses tools: what new **signal** is fed in at each stage, and how the training signal moves from human hands into the environment's hands.

We give a model that has only pretrained ability the prompt: "12 plus 8 equals?"

It does not answer 20. It continues the sentence, perhaps with "equals what", or with "let us compute it together".

The reason is that pretraining only taught it to continue a sentence. It does not know this is a problem waiting for an answer.

To turn a model from "able to continue text" into "able to do work", people ran a large amount of training after pretraining: first they used human-written question-answer examples to teach it to imitate, then annotators ranked its replies, and later they trained it with verifiable answers, even with results from environment execution. Pretraining supplies the ability to **continue text**. The training that happens after pretraining and makes the model more usable is collectively called **post-training**.

This lecture is that path. The four training stages each have a name — **SFT**, **RLHF**, **RLVR**, Agent post-training — and we look at what signal each stage feeds the model. After this lecture, we can say what problem each kind of signal solves, and what the names SFT, RLHF, and RLVR refer to. The first section starts at the origin: what a pretrained-only model outputs when it faces an instruction.

## 1. What post-training is: from continuator to instruction-following model

This section makes one claim concrete. We have said repeatedly that "pretraining only taught the model to continue text". Here we look at what "does not follow instructions" actually looks like. The pretrained model's native form is a text continuator.

A model with only pretrained ability does one thing: given any prefix, it outputs the next character that most resembles "the continuation of internet text". If an instruction is concatenated as the prefix, it continues the instruction as well, rather than treating it as a task to execute. The pretrained model does not lack capability; it lacks direction. Its optimization target is not user intent.

From continuator to an Agent that can complete tasks, the training signal changes source four times across four stages. The table below is the map for the whole lecture. Each row records two things: what the model learns from which signal, and why that signal is replaced in the next stage.

| Stage | Signal source | Data form | Objective | Representative work |
|---|---|---|---|---|
| SFT | Human demonstration | (instruction, desired reply) | Learn how to act | FLAN, InstructGPT-SFT |
| RLHF | Human preference | Ranking of candidate replies | Learn what is good | InstructGPT, ChatGPT |
| RLVR | Verifiable correctness | Rule-based criterion | Learn what is correct | DeepSeek-R1, DAPO |
| Agent post-training | Environment execution result | Full trajectory + success/failure | Learn which action sequence completes the task | RLEF, WebRL, MiRA |

We first use a minimal continuator to build the intuition that the model does not follow instructions.

Each row of the map corresponds to one kind of "data fed to the model". Different signal sources produce different sample shapes. Writing one sample for each of the four stages puts the difference on the page:

| Stage | Shape of one sample | Signal the model learns from this sample |
|:---|:---|:---|
| SFT | Instruction: "Translate this sentence into French: hello" + a user-written translation | Write what the demonstration wrote |
| RLHF | Four candidate replies to the same instruction + a ranking: reply 2 best, reply 4 next | Which reply is more preferred |
| RLVR | Problem "37 + 48" + ground-truth 85; criterion: whether the answer equals 85 | Objective correctness |
| Agent post-training | One trajectory: call search → read the return → execute code → return a result, with success or failure at the end | Whether this action sequence can complete the task |

Look at the last column. SFT gives "what should be written", RLHF gives "which one is better", RLVR gives "whether the answer is right", and Agent post-training gives "whether this action sequence works". The signal moves from "written by humans" all the way to "judged by the environment". The migration in this lecture's main thread is the migration of this column of signals: the objective the model optimizes moves from imitating human text, to preference, then to verifiable correctness, and finally to environment execution results.

We now return to the starting point and look at what a continue-only model outputs when it faces an instruction.


In [ ]:
import numpy as np
from collections import defaultdict

# Mini corpus: ordinary conversation only, no instruction-following format
corpus = ["the weather is nice we went for a walk", "what is for dinner i want noodles", "todays work is done rest early"]

# Character-level bigram counts: how often each character is followed by each other character
cnt = defaultdict(lambda: defaultdict(int))
alphabet = set()
for sent in corpus:
    for ch in sent:
        alphabet.add(ch)
    for ch, nxt in zip(sent, sent[1:]):
        cnt[ch][nxt] += 1
alphabet = sorted(alphabet)


def next_char(ch):
    """Sample the next character from the conditional distribution after ch."""
    if ch not in cnt or len(cnt[ch]) == 0:
        return np.random.choice(alphabet)
    options = list(cnt[ch].keys())
    probs = np.array([cnt[ch][c] for c in options], dtype=float)
    probs /= probs.sum()
    return np.random.choice(options, p=probs)


def continue_text(prefix, length=24):
    """Continue length characters starting from prefix."""
    out = list(prefix)
    ch = out[-1]
    for _ in range(length):
        nxt = next_char(ch)
        out.append(nxt)
        ch = nxt
    return "".join(out)


np.random.seed(42)
prompt = "User: please compute 12 plus 8 equals?"
print("Input instruction:", prompt)
print("Model continuation:", continue_text(prompt))
print()
print("Key observation: the continuator did not answer with a number; it continued the instruction as dialogue-like text.")


The continuation above can be unpacked. The last character of the prompt is "?", and "?" never appears as the successor of any character in the corpus, so next_char never enters the count table and can only sample uniformly from the whole alphabet. From that character onward, continuation follows adjacent-character statistics in the corpus.

The output is therefore a concatenation of fragments such as "rest", "want noodles", "rest early", "work is done", which are adjacent-character combinations from the three sentences. Characters at sentence ends (k, s, y) have no successor in the corpus; hitting them also falls back to uniform sampling, so the output jumps frequently.

The model did not answer "20". The instruction-execution pattern never appears in the corpus: all three sentences are declarative, and none of them teaches "after a user question, output an answer". The model is not unable to compute 12+8; its optimization target does not include answering questions.

This is the difference between a continuator and an assistant. The continuator optimizes "the next character that most resembles the corpus"; an assistant needs to optimize "what the user wants". The latter depends on the four signals in the section 1 map.


## 2. SFT and RLHF: alignment in the Chatbot era

The previous section showed that a continuator only writes onward when it faces an instruction, and does not answer. This section fills that gap: we teach it to answer. The method is to teach "how to answer", so that the model's behavior matches human expectations. That process is called alignment. There are two teaching methods, and this section develops each one.

The first method feeds human-written question-answer examples directly to the model and has it imitate. This training is called `SFT` (supervised fine-tuning). We collect a batch of (instruction, desired reply) pairs and fine-tune the model with cross-entropy, raising the probability of the demonstrated output. SFT does only this: it introduces no judgment of "good or bad". Replies that never appeared in the demonstrations are not learned, and the model cannot surpass the demonstrator's level.

The second method first has the model produce several candidates for the same instruction, asks annotators to rank those candidates, then trains the model to select the higher-scoring replies. This training is called `RLHF` (reinforcement learning from human feedback). Concretely, we first train an `RM` (reward model) to score replies, then use reinforcement learning to maximize the RM score, while a KL term constrains the policy from drifting too far from the SFT model. The RM score is a scalar proxy for annotated preference and can only measure "how good this text is".

Below we use a set of toy data to compute the SFT cross-entropy and the RLHF objective term by term, and watch where the same probabilities are pushed. The loss forms of RLVR and Agent post-training are left for section 3, where we compare them together.

To actually read the numbers in the code that follows, we first align two basic concepts.

The first is the loss function. Training a model means adjusting parameters θ so that some loss L(θ) becomes smaller. L is a number for "how well the model currently performs": the closer the behavior is to what we want, the smaller L is. Gradient descent repeatedly adjusts θ in the direction that makes L smaller. Different training objectives differ in what L looks like and where the signal inside L comes from.

The second is logits and softmax. The model does not output probabilities directly. It outputs an unnormalized score, the logits, for each token in the vocabulary, then softmax turns those scores into probabilities. Softmax raises the probability of the token with the largest logit to the highest, but the probabilities of all tokens always sum to 1. A large probability means the model is more inclined to output that token.

We use a toy setting with only 4 tokens (A, B, C, D) and compute each of the three objectives once. Initial logits are all 0, so after softmax each token has probability 0.25. The three signals are: the demonstration token is B; RM scores [0.5, 1.0, -0.2, 0.0]; correctness [0, 1, 0, 1] (B and D are correct). The same initial model and the same data yield three different gradients under the three objectives.


In [ ]:
import numpy as np


def softmax(x):
    """Normalize logits along the last axis into a probability distribution."""
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


# Toy setting: one instruction x; the model's output vocabulary has only 4 tokens
tokens = ["A", "B", "C", "D"]
logits = np.zeros(4)                          # initial logits all 0 → uniform
probs = softmax(logits)

# Three kinds of labels, corresponding to three eras of supervision
demo_token = "B"                              # SFT: the human demonstration wants reply B
rm_scores = np.array([0.5, 1.0, -0.2, 0.0])   # RLHF: RM scores for the four candidates
correctness = np.array([0.0, 1.0, 0.0, 1.0])  # RLVR: B and D are correct

print("token        :", tokens)
print("current π    :", np.round(probs, 3))
print("SFT demo token:", demo_token)
print("RLHF RM scores:", rm_scores)
print("RLVR correctness:", correctness)


The SFT loss is the cross-entropy of the output sequence. In the single-token setting it reduces to

$$L_{\mathrm{SFT}} = -\log \pi_\theta(y_{\mathrm{demo}}).$$

The gradient only pushes up the probability of the demonstration token. Below we scan the demonstration token's probability from 0.1 to 0.9, observe how the loss changes with probability, then use numerical differences to see the gradient direction.


In [ ]:
import numpy as np

demo_idx = 1  # token B

# Scan the demonstration token's probability and watch how the SFT loss changes
print("demo prob p(B) | SFT loss -log p")
for p in np.arange(0.1, 0.95, 0.1):
    print(f"      {p:.1f}      |   {-np.log(p):.4f}")
print()


def sft_loss(theta):
    """SFT cross-entropy loss in the single-token setting."""
    return -np.log(softmax(theta)[demo_idx])


theta = np.zeros(4)
eps = 1e-4
grad = np.array([(sft_loss(theta + eps * np.eye(4)[j]) -
                  sft_loss(theta - eps * np.eye(4)[j])) / (2 * eps)
                 for j in range(4)])
print("SFT gradient dL/dθ :", np.round(grad, 3))
print("Key observation: the gradient on demonstration token B is negative (probability is pushed up); gradients on the other tokens are positive (probability is pushed down).")


We unpack the hand calculation of the SFT loss so that every number has a source.

The intuition of cross-entropy: -log p tends to 0 as p approaches 1, and tends to positive infinity as p approaches 0. The closer the demonstration token's probability is to 1, the smaller the loss. Substituting p(B)=0.25: 0.25 = 1/4, log(1/4) = -1.386, so L = -log 0.25 = 1.386.

The gradient can be obtained analytically. Differentiating L = -log p(B) and using the derivative of softmax gives

$$\frac{\partial L}{\partial \theta_j} = p_j - \mathbb{1}\{j = B\}.$$

The initial p values are all 0.25, so the gradient is [0.25, -0.75, 0.25, 0.25]. The gradient on demonstration token B is negative; the others are positive. Gradient descent updates by θ ← θ - lr·dL/dθ. Taking one step with learning rate 1:

| token | initial p_j | gradient | updated θ | updated p_j |
|:---|:---|:---|:---|:---|
| A | 0.25 | +0.25 | -0.25 | 0.175 |
| B | 0.25 | -0.75 | +0.75 | 0.475 |
| C | 0.25 | +0.25 | -0.25 | 0.175 |
| D | 0.25 | +0.25 | -0.25 | 0.175 |

After one step, p(B) rises from 0.25 to about 0.475, and the other three fall to about 0.175. The numerical differences in the code above confirm this gradient.

Why this design. Cross-entropy measures "the negative log of the total probability the model assigns to the demonstration text". Driving it as low as possible is equivalent to maximizing the demonstration's probability. Note that SFT only pushes B up: it cannot see extra information such as "B and D are both actually correct", and it cannot see any good/bad judgment. Content that never appeared in the demonstration is neither learned nor known to be good or bad. That is the ceiling of SFT, and the direct reason the next stage introduces preference.


RLHF training has two steps. The first trains an RM from annotated rankings: for a set of candidates under the same instruction, a pairwise ranking loss makes the preferred reply score higher. The second treats the RM score as a reward and optimizes the model, while a KL term constrains the policy from drifting too far from a reference policy. The overall objective to minimize is

$$L_{\mathrm{RLHF}} = -\mathbb{E}_{y\sim\pi}\big[r_\theta(x,y)\big] + \beta\, D_{\mathrm{KL}}\big(\pi \,\|\, \pi_{\mathrm{ref}}\big).$$

Candidates with high RM scores are pushed up; the KL term pulls the policy back toward the reference, preventing the model from exploiting the RM and walking too far.


In [ ]:
import numpy as np

rm_scores = np.array([0.5, 1.0, -0.2, 0.0])
beta = 0.5
ref = np.full(4, 0.25)  # reference policy: uniform


def rlhf_loss(theta):
    """RLHF objective: negative expected RM score + KL constraint."""
    p = softmax(theta)
    reward = rm_scores @ p
    kl = (p * (np.log(p) - np.log(ref))).sum()
    return -reward + beta * kl


theta = np.zeros(4)
print("logit θ(B) | p(B)   | E[r]   | β·KL   | RLHF loss")
for tb in [-1.5, -0.5, 0.0, 0.5, 1.5]:
    th = theta.copy()
    th[1] = tb
    p = softmax(th)
    kl = (p * (np.log(p) - np.log(ref))).sum()
    print(f"  {tb:+.1f}   | {p[1]:.3f} | {rm_scores@p:.3f} | {beta * kl:.3f} | {rlhf_loss(th):.4f}")

eps = 1e-4
grad = np.array([(rlhf_loss(theta + eps * np.eye(4)[j]) -
                  rlhf_loss(theta - eps * np.eye(4)[j])) / (2 * eps)
                 for j in range(4)])
print()
print("RLHF gradient dL/dθ :", np.round(grad, 3))
print("Key observation: tokens whose RM score is above the mean (0.325) are pushed up; those below the mean are pushed down; the KL term pulls the whole distribution back toward uniform.")


We compute each of RLHF's two steps once.

The first step trains the RM. For a set of candidates under the same instruction, annotators pick the more preferred one. The RM is trained with a pairwise ranking loss: for a pair (y_w, y_l), the preferred y_w is required to score clearly higher than y_l. The loss is

$$L_{\mathrm{RM}} = -\log\sigma\big(r(x, y_w) - r(x, y_l)\big).$$

We try two pairs on the toy RM scores. (B, C): 1.0 - (-0.2) = 1.2, σ(1.2) ≈ 0.769, loss = -log 0.769 ≈ 0.263. (A, D): 0.5 - 0.0 = 0.5, σ(0.5) ≈ 0.622, loss ≈ 0.474. The larger the score gap, the smaller the loss: the RM is trained to "give higher scores to more preferred replies".

The second step treats RM scores as rewards and optimizes the policy. The objective is

$$L_{\mathrm{RLHF}} = -\mathbb{E}_{y\sim\pi}\big[r(x,y)\big] + \beta\, D_{\mathrm{KL}}\big(\pi \,\|\, \pi_{\mathrm{ref}}\big).$$

The first term is the negation of the expected RM score: the more probability flows to high-scoring tokens, the smaller this term. The second term is KL divergence, which penalizes the policy for leaving the reference π_ref; the farther the model drifts, the larger the KL, and β controls the penalty strength. It prevents the model from walking too far in order to farm RM scores.

Hand calculation of the gradient at the initial point. π_ref is uniform, and the initial p is also uniform, so KL = 0. The expected RM score E[r] = (0.5 + 1.0 - 0.2 + 0.0)/4 = 0.325. Differentiating L and substituting p = 0.25 gives

$$\frac{\partial L}{\partial \theta_k} = -p_k\big(r_k - \mathbb{E}[r]\big).$$

| token | r_k | r_k - 0.325 | gradient | updated p_k |
|:---|:---|:---|:---|:---|
| A | 0.5 | +0.175 | -0.044 | 0.259 |
| B | 1.0 | +0.675 | -0.169 | 0.294 |
| C | -0.2 | -0.525 | +0.131 | 0.218 |
| D | 0.0 | -0.325 | +0.081 | 0.229 |

A and B, whose gradients are negative, are pushed up; C and D, whose gradients are positive, are pushed down. After one step and a fresh softmax, B's probability rises to 0.294, the highest; A is next at 0.259; C is lowest at 0.218. The scan table and numerical differences in the code above confirm these numbers.

The difference from SFT is already clear here. SFT's signal is "the demonstrated answer is B", so only B is pushed up; RLHF's signal is "the relative preference among four candidates", so the whole distribution is reallocated by RM score. SFT learns to imitate the demonstration; RLHF learns to rank by preference. The model actively pushes down low-scoring answers, even if they also appeared in demonstrations.


The RM in RLHF is a proxy objective, and the model may game it. If the RM only counts whether a keyword appears in the reply, the model will pile up keywords to get a high score, and those texts have no value for the user. That is `reward hacking`. The RM can also only measure text quality; it cannot measure "whether this action sequence can complete the task". Constitutional AI tried to compress human annotation cost by letting an AI critique and revise itself against a list of principles. Lecture 04 already read that paper closely, so we do not expand it here.

These two limits are the direct motive for changing the signal source in the next stage: rather than learning a reward function, check whether the answer is correct.


We make RLHF's two limits concrete with one example each.

First, the RM is a proxy objective and may be gamed by the model. Suppose high-scoring replies in the RM training data commonly contain "first", "second", "therefore". The RM then treats "the presence of such words" as a signal of a good reply. Once the model discovers this, it piles up those words to farm scores, and the text has no practical value for the user. The model optimizes "make the RM assign a high score", not "actually complete the task". That phenomenon is called reward hacking.

Second, the RM can only measure text quality, not task success or failure. Let the model solve a math problem: an RM that sees a reply with "complete reasoning and a confident tone" may assign a high score, but if the answer is wrong, that high score is a wrong signal. The ground-truth answer is not in the RM's input, so the RM has no way to check.

Both point in the same direction: rather than having the model guess "what humans like", tell it directly "what is correct". Tasks for which a criterion can be written — answers with a ground truth, code with tests — can be checked directly. RLVR in the next section turns this idea into a training objective.


## 3. Agent-style post-training: tools, execution, and feedback

The previous section's RLHF relies on annotators to assign scores. Humans are expensive, and the RM may be gamed by the model. This section treats a definite-answer task: scoring need not rely on humans. We can use a rule to judge right or wrong. We first cover using a rule as the reward, then walk step by step to Agent post-training, where the environment is the judge.

RLHF's reward model is learned, and the model may game it. When a criterion can be written for the task, we can drop that learned reward model and let a rule judge directly: 1 if the answer matches the ground truth, 0 otherwise; 1 if the code passes all tests, 0 otherwise. Reinforcement learning with a verifiable reward is called `RLVR` (reinforcement learning with verifiable rewards). DeepSeek-R1's R1-Zero starts from the base model, does no SFT, and trains with pure RLVR. Long-chain reasoning emerges on its own: the model learns to think before answering, because thinking more makes it easier to score. As long as the reward is correct and verifiable, a capable policy does not need human demonstration.

RLVR's optimization uses in-group advantage: sample a group of candidates for the same instruction, subtract the group mean from each candidate's reward, and divide by the group standard deviation. That is the core of `GRPO`; it does not need a value network. We first compute this set of normalized advantages by hand, then place the three eras' loss forms side by side and watch where each one pushes probability.

The RLVR reward is a criterion, not a neural network. Examples of a criterion: 1 if the answer matches the ground truth, otherwise 0; 1 if the code passes all tests, otherwise 0. Once the criterion is written, any candidate can be scored at no extra labeling cost, which is much cheaper than training an RM.

GRPO uses one device to turn "how high the reward is" into "how high relative to the group". Sample a group of candidates for the same instruction, subtract the group mean from each reward, divide by the group standard deviation, and obtain the in-group advantage. We compute one group by hand. Suppose an instruction yields 4 sampled candidates with rewards [1, -1, 0, 1]:

group mean = (1 + (-1) + 0 + 1)/4 = 0.25
deviation = (0.75, -1.25, -0.25, 0.75)
variance = (0.75² + 1.25² + 0.25² + 0.75²)/4 = 2.75/4 = 0.6875
standard deviation = √0.6875 ≈ 0.829
advantage = deviation/0.829 ≈ (0.905, -1.508, -0.302, 0.905)

The highest-reward candidate receives about +0.9; the lowest receives about -1.5. The code below computes this set of numbers as written.

Why subtract the mean and divide by the standard deviation. Subtracting the mean picks out "who in this group is above average": above average gets a positive advantage, below average a negative advantage. Dividing by the standard deviation unifies scale, so signals from different groups and different reward units are comparable. What the model learns is "who is better inside the group", not "whose absolute score is high". The cost is here as well: when a group is all correct or all incorrect, the mean is 1 or -1, every advantage after subtracting the mean is 0, and this group of samples produces no gradient.


In [ ]:
import numpy as np


def group_advantage(rewards, eps=1e-9):
    """In-group normalized advantage; (r - mean) / (std + eps) avoids division by zero on all-correct/all-incorrect groups."""
    return (rewards - rewards.mean()) / (rewards.std() + eps)


rewards = np.array([1.0, -1.0, 0.0, 1.0])
mean = rewards.mean()
std = rewards.std()
advantage = group_advantage(rewards)

print("reward r         :", rewards)
print("group mean       :", round(mean, 3))
print("group std        :", round(std, 3))
print("in-group advantage:", np.round(advantage, 3))
print("Key observation: correct candidates (1.0) receive a positive advantage and are pushed up; incorrect candidates receive a negative advantage and are pushed down.")

# How the GRPO objective changes with the probability of a correct token
print()
print("p(D) | GRPO weighted log-likelihood objective")
for p in [0.1, 0.3, 0.5, 0.7, 0.9]:
    pi = np.array([(1 - p) / 3] * 3 + [p])
    adv = group_advantage(np.array([0.0, 1.0, 0.0, 1.0]))
    obj = -(adv * np.log(pi)).sum() / 4
    print(f" {p:.1f} |        {obj:.4f}")

# All-correct and all-incorrect groups: advantage is identically 0
print()
for name, r in [("all-correct group", np.array([1.0, 1.0, 1.0, 1.0])),
                ("all-incorrect group", np.array([-1.0, -1.0, -1.0, -1.0]))]:
    adv = group_advantage(r)
    print(f"{name} r={r.tolist()} -> advantage={np.round(adv, 3)}")
print("Key observation: when there is no within-group difference, every advantage is 0, and this group of samples produces no gradient.")


We now place the three objectives on the same batch of data and compute them side by side. The data is the earlier toy: 4 tokens A, B, C, D, initial probabilities all 0.25. The three signals are: SFT's demonstration token is B; RLHF's RM scores [0.5, 1.0, -0.2, 0.0]; RLVR's correctness [0, 1, 0, 1] (B and D are correct).

The gradient formulas of the three losses were each derived earlier:

SFT:  ∂L/∂θ_j = p_j - 1{j = demo}
RLHF: ∂L/∂θ_k = -p_k (r_k - E[r])
RLVR: ∂L/∂θ_j = -advantage_j / 4

Substituting the initial p = 0.25, the three gradients are:

| Objective | gradient dL/dθ (A, B, C, D) | who is pushed up |
|:---|:---|:---|
| SFT | (+0.25, -0.75, +0.25, +0.25) | only demonstration B |
| RLHF | (-0.04, -0.17, +0.13, +0.08) | A and B, above the mean score |
| RLVR | (+0.25, -0.25, +0.25, -0.25) | correct B and D |

Taking one step of each with learning rate 1, then softmax, the three models that started from the same initial point have probabilities:

| Objective | probability after one step (A, B, C, D) |
|:---|:---|
| SFT | (0.175, 0.475, 0.175, 0.175) |
| RLHF | (0.259, 0.294, 0.218, 0.229) |
| RLVR | (0.189, 0.311, 0.189, 0.311) |

Three models look at the same data and walk in different directions. The difference comes entirely from the signal.

The most notable split is token D. RLVR treats D as correct (D is 1 in the correctness array) and raises D's probability to tie with B. In RLHF, humans scored D as 0, below the mean 0.325, so RLHF instead pushes D down. The same token is objectively a correct answer, yet is not preferred. The two signals give opposite update directions.

Putting the three rows together makes clear what "migration of the signal source" actually migrates. SFT knows what was written in the demonstration, and only pushes B. RLHF knows whom humans think is better, and rearranges the whole distribution by score. RLVR knows objective right or wrong, and pushes every correct token, whether humans like it or not. What migrates is the signal the model optimizes: from "one correct answer", to "relative preference", to "objective correctness". The code below uses numerical differences to confirm these three gradient vectors.


In [ ]:
import numpy as np

demo_idx = 1
rm_scores = np.array([0.5, 1.0, -0.2, 0.0])
correctness = np.array([0.0, 1.0, 0.0, 1.0])
beta = 0.5
ref = np.full(4, 0.25)


def sft_loss(theta):
    """SFT cross-entropy."""
    return -np.log(softmax(theta)[demo_idx])


def rlhf_loss(theta):
    """RLHF objective: negative expected RM score + KL constraint."""
    p = softmax(theta)
    return -(rm_scores @ p) + beta * (p * (np.log(p) - np.log(ref))).sum()


def grpo_loss(theta):
    """Log-likelihood weighted by in-group advantage (an equivalent objective for the policy gradient)."""
    p = softmax(theta)
    adv = group_advantage(correctness)
    return -(adv @ np.log(p)).sum() / len(correctness)


def grad_of(loss, theta):
    """Gradient by numerical differences."""
    eps = 1e-4
    return np.array([(loss(theta + eps * np.eye(4)[j]) -
                      loss(theta - eps * np.eye(4)[j])) / (2 * eps)
                     for j in range(4)])


theta = np.zeros(4)
for name, loss in [("SFT", sft_loss), ("RLHF", rlhf_loss), ("RLVR/GRPO", grpo_loss)]:
    g = grad_of(loss, theta)
    direction = ["↑ raise" if x < 0 else "↓ lower" for x in g]
    print(f"{name:9s} gradient {np.round(g, 3)}  direction {direction}")
print()
print("Key observation: SFT only pushes up demonstration token B; RLHF pushes up high-RM-score A and B; RLVR pushes up correct B and D.")


The three gradients printed by the code match the hand calculation: SFT only pushes B; RLHF pushes A and B, and lowers C and D; RLVR pushes B and D, and lowers A and C. The same initial model and the same data, three objectives, three update directions. The gradient is the objective telling the model "which way to go".

This static comparison looked at only one step. Real training is dynamic: after every update the model resamples, the criterion or environment rescores, and the next batch of data arrives. SFT sees a fixed demonstration set, RLHF sees fixed RM scores, and RLVR sees a group of rewards that change with sampling. To see convergence behavior we have to actually run a training loop. The next section implements PPO and GRPO on the same toy bandit, and compares how the two advantage estimators behave under dynamic updates.


The comparison above is static: given a set of probabilities, look at the gradient direction. Real training is dynamic. Every update, the model resamples and the environment rescores. Below we implement `PPO` (proximal policy optimization) and GRPO on a toy bandit, and observe how the two advantage estimators affect convergence.

PPO is the reinforcement learning method used most widely to train chat models. At every update it constrains the new policy from leaving the old policy too far, to avoid one update ruining the model, hence "proximal". GRPO is the method used by the DeepSeek series that lecture 06 read; it drops the value network inside PPO.

The setting is a contextual bandit: 5 problems, each with a unique correct action (10 candidates in total). The environment judges actions right or wrong by a rule, with reward +1/-1, and there is no intermediate layer that can be hacked. PPO uses a value network to estimate a baseline; GRPO uses the in-group mean as the baseline. Both have to answer the same thing: raise the probability of the correct action for every problem.

PPO and GRPO are both based on the same policy-gradient update: after sampling action a, push a's logit in the direction of higher advantage. Written as

$$\logits[a] \leftarrow \logits[a] + \alpha A (1 - p(a)), \qquad \logits[j \neq a] \leftarrow \logits[j] - \alpha A\, p(j).$$

A is the advantage. If A is positive, action a's logit rises and the other actions' logits fall; if A is negative, the opposite. One sample gives only one (problem, action, reward) triple, and A is the entire signal this triple gives the policy.

PPO and GRPO differ only in where A comes from. PPO uses a value network: each problem keeps a scalar baseline value[q], A = r - value[q]. During training, value[q] continually regresses toward "the expected reward of that problem". In this toy, when each problem is sampled uniformly the probability of picking the correct action is 1/10, so value[q] converges to 1×(1/10) + (-1)×(9/10) = -0.8; then the advantage of a correct action is 1 - (-0.8) = 1.8, and of an incorrect action is -1 - (-0.8) = -0.2. Advantage measures "how much better this actual reward is than usual".

GRPO does not use a value network. A is computed from the other candidates' rewards in the same group: A_i = (r_i - group mean)/group std. Dropping the extra value network has a cost: each group must sample several candidates, and all-correct/all-incorrect groups have every advantage equal to 0, so the gradient is empty.

One easy confusion: the mean of advantages is not the same as the mean of the raw rewards r. The mean of r is "how many points this batch scored on average"; the mean of advantages is identically 0, because subtracting the mean recenters at the group average. The policy gradient cares about relative high and low, not absolute scores.


In [ ]:
import numpy as np

rng = np.random.default_rng(42)

NUM_Q = 5
NUM_A = 10
correct = rng.integers(0, NUM_A, size=NUM_Q)   # correct action for each problem

logits = np.zeros((NUM_Q, NUM_A))   # policy parameters
value = np.zeros(NUM_Q)             # value network: one scalar baseline per problem
lr_policy = 0.05
lr_value = 0.1


def sample_one():
    """Sample one (problem, action) and return problem, action, reward, and sampling probability."""
    q = int(rng.integers(0, NUM_Q))
    p = softmax(logits[q])
    a = int(rng.choice(NUM_A, p=p))
    r = 1.0 if a == correct[q] else -1.0
    return q, a, r, p


def ppo_step():
    """Simplified PPO: advantage = reward - value baseline; the policy is updated weighted by advantage."""
    for _ in range(32):
        q, a, r, p = sample_one()
        adv = r - value[q]
        logits[q] += lr_policy * adv * (np.eye(NUM_A)[a] - p)
        value[q] += lr_value * (r - value[q])   # value regresses toward the reward


ppl_acc = []
for it in range(80):
    ppo_step()
    p = softmax(logits)
    acc = (np.argmax(p, axis=1) == correct).mean()
    ppl_acc.append(acc)

print("PPO accuracy (every 20 rounds):", [round(float(x), 2) for x in ppl_acc[::20]])
print("PPO late-stage probability of the correct action for each problem:",
      np.round(softmax(logits)[np.arange(NUM_Q), correct], 2))


In [ ]:
import numpy as np

rng = np.random.default_rng(7)

# Same set of problems as PPO; reinitialize the policy
logits_g = np.zeros((NUM_Q, NUM_A))
lr_g = 0.2
K = 4  # sample 4 actions per group


def grpo_update(q):
    """GRPO: sample a group of actions for one problem, normalize advantage within the group, no value network."""
    p = softmax(logits_g[q])
    acts = rng.choice(NUM_A, size=K, p=p)
    rewards = np.array([1.0 if a == correct[q] else -1.0 for a in acts])
    adv = group_advantage(rewards)
    for i, a in enumerate(acts):
        onehot = np.zeros(NUM_A)
        onehot[a] = 1.0
        logits_g[q] += lr_g / K * adv[i] * (onehot - p)
    return rewards


g_acc = []
wasted_history = []
for it in range(80):
    wasted = 0
    for _ in range(8):   # 8 groups per round
        q = int(rng.integers(0, NUM_Q))
        rewards = grpo_update(q)
        if rewards.max() == rewards.min():
            wasted += 1
    wasted_history.append(wasted)
    p = softmax(logits_g)
    g_acc.append((np.argmax(p, axis=1) == correct).mean())

print("GRPO accuracy (every 20 rounds):", [round(float(x), 2) for x in g_acc[::20]])
print("Number of all-correct/all-incorrect groups per round (first 10 rounds):", wasted_history[:10])
print("Key observation: GRPO needs no value network, but all-correct and all-incorrect groups produce no gradient; those samples are wasted compute.")

import matplotlib.pyplot as plt

plt.figure(figsize=(6.2, 3.8))
plt.plot(ppl_acc, label="PPO (critic baseline)")
plt.plot(g_acc, label="GRPO (group baseline)")
plt.xlabel("round")
plt.ylabel("accuracy")
plt.title("Correct-arm accuracy on a toy bandit")
plt.legend()
plt.tight_layout()
plt.show()


RLVR solved "whether the text is good", but it only applies to tasks for which a criterion can be written: math, code, puzzles. Open-ended problems (write a well-mannered email) have no ground-truth answer, and therefore no verifier. The key transfer in Agent-style post-training is: the task may be open-ended, but the environment itself can serve as a verifier. Task success can be judged by whether tests pass, whether a goal is reached, or whether a terminal state has converged.

The training unit also changes from a piece of text to a whole trajectory. The action space expands from the next token to a sequence of tool calls: query a database, execute code, write a file, return a result. The reward is no longer a criterion written before the task starts, but a result the environment runs — possibly sparse, and possibly delayed until the end of the trajectory. We first place the four stages' signal sources on one figure and watch how cost and reliability change along the way.


The most important sentence in Agent post-training is: the task may be open-ended, but the environment itself can serve as a verifier. We look at that sentence on concrete tasks.

Writing a well-mannered email is an open-ended task: there is no ground-truth answer, and no criterion can be written. But "use search to finish a report and output a file" can be verified: whether the file was produced, whether the format meets the spec, whether key data appear, even whether a test script runs. Success on an open-ended task is judged by the execution result, not preset by a person.

The training unit changes with it. Samples from SFT through RLVR are all text: one reply, or the answer to one problem. Agent post-training has to learn an action sequence, so a sample is a trajectory: call search, read the return, execute code, write a file, return a result. The action space expands from "the next token" to "a sequence of tool calls".

The signal becomes sparse and delayed. A criterion reward scores as soon as the action ends; an Agent's reward waits until the whole trajectory has run, and only then does the environment know success or failure, perhaps a +1 or -1 dozens of steps later. The next figure projects the four stages' signals onto the (annotation cost, resistance to hacking) plane, and we watch the direction of the migration.


In [ ]:
import matplotlib.pyplot as plt

# Four signal sources, projected onto the (annotation cost, resistance to hacking) plane
signals = [
    ("SFT human demo", 1.0, 0.30),
    ("RLHF human pref", 0.8, 0.50),
    ("RLVR rule", 0.4, 0.90),
    ("Agent RL env", 0.2, 1.00),
]
names = [s[0] for s in signals]
costs = [s[1] for s in signals]
rel = [s[2] for s in signals]

plt.figure(figsize=(6.2, 4.4))
plt.scatter(costs, rel, s=260, c=range(4), cmap="viridis")
for i, name in enumerate(names):
    plt.annotate(name, (costs[i], rel[i]),
                 textcoords="offset points", xytext=(6, 4), fontsize=9)
plt.xlabel("annotation cost (lower is cheaper)")
plt.ylabel("resistance to reward hacking")
plt.title("Reward signal sources across post-training stages")
plt.xlim(0, 1.2)
plt.ylim(0, 1.2)
plt.tight_layout()
plt.show()

print("Key observation: the signal source migrates from upper-right to lower-left — annotation cost falls along the way, reliability rises along the way.")


A trajectory-level reward compresses the whole trajectory into a single +1 or -1. If a trajectory has dozens of steps and only the last step carries a signal, the model cannot tell which intermediate step went wrong. That is the `credit assignment` problem, and the core difficulty of sparse delayed reward. Step-level (milestone) rewards split the total reward across steps: a share is given each time a subgoal is reached, and another share at the end if the task succeeds, so the model receives a process signal.

Below we numerically compare gradient variance and convergence speed of the two rewards in a toy three-step environment. Trajectory reward spreads the same terminal value across every step, and variance is amplified; milestone reward carries only its own subgoal signal at each step, and variance is smaller.


We unpack the credit assignment problem with concrete numbers. A three-step task (query a database → filter → return a result); each step chooses among 3 actions; success requires all three steps correct.

First the success rate. Under a uniform policy the probability of choosing correctly at each step is 1/3, so the probability of all three correct is (1/3)³ = 1/27 ≈ 0.037. Under trajectory reward, about 1 in 27 trajectories receives +1, and the other 26 receive -1. If a trajectory got the first step right and the later ones wrong, it receives the same -1 as "all three wrong". The model has no way to tell "the first step was actually correct".

The gradient makes this clearer. The per-step update of the policy gradient is

$$g_t = R\,(\mathrm{onehot}(a_t) - p_t),$$

where R is the return of this trajectory. Under trajectory reward, the same R (+1 or -1) scales all three steps' gradients at once. Milestone reward replaces R with the per-step subgoal signal r_t (in this toy +0.3 per step, plus +0.1 at the end on success). The first step, if correct, receives +0.3 on its own, independent of later success or failure.

The difference in variance can be quantified. Under a uniform policy, looking at one component of the gradient: trajectory reward has R² = 1, so gradient variance ≈ 2/9 ≈ 0.22; milestone reward carries a signal only on the step that was chosen correctly, so gradient variance ≈ 0.007. The former is about 30 times the latter. Large variance means the gradient direction fluctuates more for the same number of samples, and convergence is slower. Milestones split the signal by step, and every step has a definite process signal. The code below measures gradient variance and convergence curves of the two rewards in the same environment.


In [ ]:
import numpy as np

rng = np.random.default_rng(1)
N_ACT = 3
correct_steps = np.array([0, 1, 2])   # the correct choice at step 3 decides terminal success


def sample_trajectory(logits):
    """Sample a three-step action sequence from the policy."""
    return np.array([rng.choice(N_ACT, p=softmax(logits[t])) for t in range(3)])


def trajectory_reward(acts):
    """+1 only if all three steps are correct, otherwise -1."""
    return 1.0 if (acts == correct_steps).all() else -1.0


def milestone_rewards(acts):
    """+0.3 if the subgoal at that step is reached; +0.1 more at the end on success."""
    per = 0.3 * (acts == correct_steps).astype(float)
    if (acts == correct_steps).all():
        per[2] += 0.1
    return per


def reinforce_grad(logits, mode):
    """Policy-gradient estimate from one episode."""
    acts = sample_trajectory(logits)
    g = np.zeros_like(logits)
    if mode == "trajectory":
        R = trajectory_reward(acts)
        for t in range(3):
            g[t] = R * (np.eye(N_ACT)[acts[t]] - softmax(logits[t]))
    else:
        r = milestone_rewards(acts)
        for t in range(3):
            g[t] = r[t] * (np.eye(N_ACT)[acts[t]] - softmax(logits[t]))
    return g, acts


# Start from the same uniform policy, sample 500 gradients each, compare variance
logits0 = np.zeros((3, N_ACT))
traj_grads, mile_grads = [], []
for _ in range(500):
    g1, _ = reinforce_grad(logits0, "trajectory")
    g2, _ = reinforce_grad(logits0, "milestone")
    traj_grads.append(g1)
    mile_grads.append(g2)
var_t = np.array(traj_grads).var()
var_m = np.array(mile_grads).var()
print("Gradient variance under a uniform policy: trajectory reward", round(var_t, 4), " vs milestone reward", round(var_m, 4))
print()

# Training comparison: 400 rounds, plot 50-round running mean return
M = 400
lr = 0.1


def train(mode):
    """Run M rounds of REINFORCE and return per-round terminal return."""
    logits = np.zeros((3, N_ACT))
    returns = []
    for it in range(M):
        g, acts = reinforce_grad(logits, mode)
        logits += lr * g
        returns.append(1.0 if (acts == correct_steps).all() else -1.0)
    return returns


def running_mean(x, w=50):
    """Running mean over a window of width w."""
    out = []
    for i in range(len(x)):
        lo = max(0, i - w + 1)
        out.append(np.mean(x[lo:i + 1]))
    return out


ret_t = running_mean(train("trajectory"))
ret_m = running_mean(train("milestone"))
print("Running mean return after 400 rounds: trajectory reward", round(ret_t[-1], 3),
      " vs milestone reward", round(ret_m[-1], 3))

import matplotlib.pyplot as plt

plt.figure(figsize=(6.2, 3.8))
plt.plot(ret_t, label="trajectory reward")
plt.plot(ret_m, label="milestone reward")
plt.xlabel("episode")
plt.ylabel("running mean return")
plt.title("Credit assignment: sparse vs dense reward")
plt.legend()
plt.tight_layout()
plt.show()


The code's measurements confirm the hand calculation: under a uniform policy, trajectory reward has gradient variance 0.2222, milestone reward only 0.0072, a factor of about 30; after 400 training rounds, trajectory reward's running mean return stalls around 0.16, while milestone reward rises to about 0.52. The model for which a signal is available at every step converges faster.

The term credit assignment means "attribute the success or failure of a whole trajectory to some step inside it". Trajectory reward's attribution is "every step shares the blame": on terminal failure, every step's gradient is scaled by the same minus sign, even if some step was correct. Milestone reward's attribution is "each step is responsible only for its own subgoal": the first step, if correct, receives +0.3, independent of later success or failure. The finer the attribution, the less confused the gradient signal.

Agent-task trajectories often run to dozens of steps, and the reward is sparse by nature: the environment only signals at the end of the task. MiRA's method is to split "terminal success or failure" into several decidable milestones, so the model has a process signal to learn at every step.


Putting the whole chain together yields an environment-feedback flywheel. The model outputs an action sequence → the environment executes and judges success or failure → the success/failure signal is used as a reward to update the policy → successful trajectories are filtered and we resample. That is the skeleton of STaR and WebRL. Failed tasks have two fates: they are discarded, or they are rewritten into a verifiable form and reinjected into the training set. The latter lets the data pool expand round by round with training.

Below we run this flywheel on a toy arithmetic task. The model (a scripted instance of llm_client) proposes an answer to each problem; the environment checks the answer and returns +1/-1; we update a lightweight answer policy from that. Each round we record the success rate, watch it rise round by round, and demonstrate the two fates of a failed task.


In [ ]:
import sys, os, re
import numpy as np

# Always go through llm_client.py at the repo root; the same demo can run against a real API
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "llm_client.py")):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)
from llm_client import get_llm

client = get_llm()
print("Current LLM mode:", "scripted example" if False else "real API")

# Question pool: the scripted example can add the first three items; it does not handle the last multiplication item
questions = ["compute 13 plus 24", "compute 35 plus 19",
             "compute 7 plus 46", "compute 12 times 7"]
answers = {"compute 13 plus 24": 37, "compute 35 plus 19": 54,
           "compute 7 plus 46": 53, "compute 12 times 7": 84}


def extract_number(text):
    """Extract the first integer from a reply; return None if none is found."""
    m = re.search(r"-?\d+", text)
    return int(m.group()) if m else None


def propose(q):
    """The model proposes an answer: ask the LLM once, parse out a number."""
    reply = client.chat([{"role": "user", "content": f"{q}, output the number only."}])
    return extract_number(reply)


proposal = {q: propose(q) for q in questions}

# Candidate answer pool: the scripted example's proposal + distractors; the multiplication item's correct answer is not in the pool
candidate_pool = {}
for q in questions:
    cands = []
    if proposal[q] is not None:
        cands.append(proposal[q])
    cands += [answers[q] + 3, answers[q] - 7]   # distractors
    candidate_pool[q] = list(dict.fromkeys(cands))

for q in questions:
    print(f"{q}: proposal {proposal[q]}, candidate pool {candidate_pool[q]}")
print("Key observation: the first three items' proposals are the correct answers; the multiplication item's proposal is None, and the correct answer is not in the pool.")


In [ ]:
rng = np.random.default_rng(3)
lr = 0.2
logits_map = {q: np.zeros(len(candidate_pool[q])) for q in questions}
q_fail = questions[-1]


def run_round():
    """Run one round of RL: sample an answer for each problem, the environment checks, REINFORCE updates. Return per-problem success."""
    per_q = {}
    for q, cands in candidate_pool.items():
        p = softmax(logits_map[q])
        idx = int(rng.choice(len(cands), p=p))
        a = cands[idx]
        r = 1.0 if a == answers[q] else -1.0
        logits_map[q] += lr * r * (np.eye(len(cands))[idx] - p)
        per_q[q] = (a == answers[q])
    return per_q


def correct_prob():
    """Mean policy probability of the correct answer (only problems whose correct answer is in the pool)."""
    vals = []
    for q, cands in candidate_pool.items():
        if answers[q] in cands:
            vals.append(float(softmax(logits_map[q])[cands.index(answers[q])]))
    return np.mean(vals)


def block_means(x):
    """Cut the per-round series into blocks of 15 rounds and return block means."""
    return [round(float(np.mean(x[i * 15:(i + 1) * 15])), 2)
            for i in range(len(x) // 15)]


# Do not rewrite failed tasks: 60 consecutive rounds, watch success rate and correct-answer probability
rates, probs, fail_rate = [], [], []
for it in range(60):
    per_q = run_round()
    rates.append(np.mean(list(per_q.values())))
    probs.append(correct_prob())
    fail_rate.append(per_q[q_fail])

print("Overall sample success rate (every 15 rounds):", block_means(rates))
print("Mean probability of the correct answer (every 15 rounds):", block_means(probs))
print("Multiplication-problem sample success rate (every 15 rounds):", block_means(fail_rate))
print("Key observation: the first 3 items' correct-answer probability rises block by block; the multiplication item's correct answer is not in the pool, so the success rate is identically 0 — the task is discarded.")


Note the multiplication problem's result above: the sample success rate is identically 0. The reason is not that the model cannot do it, but that the correct action is not in the candidate pool at all.

An RL update can only adjust the probabilities of "actions already in the pool". The multiplication problem's candidate pool is [87, 77] (the scripted example's proposal was None and was skipped), and the correct answer 84 is not among them. No matter how the policy is tuned, it can only allocate probability between those two, and will never select 84. That is equivalent to the action space not containing "the correct action" as an option; no reward, however good, can push the policy onto it.

There are two ways to handle a failed task. The first is to discard it: remove it from the training set, and the model never learns this problem. The second is to rewrite it: split the task into substeps the environment can verify, let the model check them one by one, and add the verified correct answer into the candidate pool. Rewriting makes a failed task trainable again. The code below demonstrates the process after rewriting: the candidate pool expands, and the correct probability rises block by block.


In [ ]:
import matplotlib.pyplot as plt

# Fate B: rewrite — split multiplication into repeated addition, use the scripted example to check each subgoal, the environment verifies the final value
acc = 12
verified = None
for step in range(6):
    target = acc + 12
    reply = client.chat([{"role": "user",
                          "content": f"check whether {acc} plus 12 equals {target}, output the number only."}])
    if extract_number(reply) != target:
        break
    acc = target
verified = acc

pool_before = sum(len(c) for c in candidate_pool.values())
print("Verified answer obtained by rewriting:", verified)
assert verified is not None, "the real model did not return a verifiable answer"

# The correct answer enters the candidate pool; the policy is reinitialized
candidate_pool[q_fail] = list(dict.fromkeys(candidate_pool[q_fail] + [verified]))
logits_map[q_fail] = np.zeros(len(candidate_pool[q_fail]))
pool_after = sum(len(c) for c in candidate_pool.values())
print("Candidate pool of the multiplication problem after rewriting:", candidate_pool[q_fail])
print("Data-pool size: before rewriting", pool_before, "→ after rewriting", pool_after)

# Continue 100 rounds of RL and watch the multiplication problem being learned as the data pool expands
rates2, prob2, fail2 = [], [], []
for it in range(100):
    per_q = run_round()
    rates2.append(np.mean(list(per_q.values())))
    prob2.append(float(softmax(logits_map[q_fail])[candidate_pool[q_fail].index(84)]))
    fail2.append(per_q[q_fail])

print("Overall sample success rate after rewriting (every 20 rounds):", block_means(rates2))
print("Multiplication-problem correct probability after rewriting (every 20 rounds):", block_means(prob2))
print("Multiplication-problem sample success rate after rewriting (every 20 rounds):", block_means(fail2))

plt.figure(figsize=(6.2, 3.8))
plt.plot(rates, label="overall (before curriculum)")
plt.plot(rates2, label="overall (after curriculum)")
plt.plot(prob2, label="correct prob of failed task")
plt.xlabel("round")
plt.ylabel("success rate / prob")
plt.title("Environment feedback flywheel")
plt.legend()
plt.tight_layout()
plt.show()


The output after rewriting shows two things. First, once the verified correct answer 84 enters the candidate pool, the multiplication problem's correct probability rises block by block from a first-block mean of 0.62 to 0.97, and the sample success rate rises with it: the task goes from "fails forever" to "learnable". Second, the data pool expands from 11 candidates before rewriting to 12 after. That is "the data pool expands round by round with training": each time a failed task is solved, a batch of verified data is added to the pool.

This is one full turn of the environment-feedback flywheel: the model outputs an action sequence → the environment executes and judges success or failure → the success/failure signal updates the policy → failed tasks are filtered or rewritten → we resample. Each turn of the flywheel, trainable data grows a little, and the model's grip on the environment grows a little. The evolution path of the four stages together is in the next section.


## 4. Evolution map: migration of the signal source

The first three sections walked through the four training signals one by one. This section places them on one timeline. The relation among the four signals is a migration along the way.

SFT in 2021-2022 used human demonstration, RLHF in 2022-2023 used human preference, RLVR in 2024-2025 used a rule-based criterion, and Agent post-training in 2024-2026 uses environment results. Each new stage did not retire the previous one; it stacked on top of it. An Agent model still needs SFT first, then alignment, then RLVR, and only then trajectory-level RL.

The figure below marks representative work of each stage on the timeline, with the signal source written under the node.

In [ ]:
import matplotlib.pyplot as plt

stages = [
    (2021.0, "SFT\nhuman demos", "FLAN / InstructGPT-SFT"),
    (2022.6, "RLHF\nhuman prefs", "InstructGPT / ChatGPT"),
    (2024.1, "RLVR\nverifiable rules", "DeepSeek-R1 / DAPO"),
    (2025.3, "Agent RL\nenvironment", "RLEF / WebRL / MiRA"),
]

fig, ax = plt.subplots(figsize=(6.6, 3.0))
ax.axhline(0, color="gray", lw=1)
for x, label, work in stages:
    ax.scatter(x, 0, s=90, zorder=3)
    ax.annotate(label, (x, 0), xytext=(0, 14), textcoords="offset points",
                ha="center", fontsize=9)
    ax.annotate(work, (x, 0), xytext=(0, -20), textcoords="offset points",
                ha="center", fontsize=7, color="dimgray")
ax.set_xlim(2020, 2026.8)
ax.set_ylim(-0.4, 0.4)
ax.axis("off")
plt.title("Post-training evolution: reward moves from humans to environment")
plt.tight_layout()
plt.show()


We map this timeline back onto the course map. Lecture 04's RLEF and Constitutional AI are the seeds of Agent feedback; lecture 06's GRPO and DAPO are the algorithm engines of Agent post-training; lecture 08's deep research is one concrete form of "the environment as verifier". This lecture gathers them into one main thread. Later, lecture 10 on SWE agents, lecture 11 on memory, and lecture 14 on evaluation will repeatedly use the three words trajectory, environment feedback, and evaluation.

In one sentence: the history of post-training is the history of the reward signal moving from human hands, into the verifier's hands, and finally into the environment's hands.


## Summary

What this lecture covered:

- [ ] A pretrained model is a continuator; an instruction is text to be continued. Post-training changes the optimization target from the next token to user intent
- [ ] SFT uses human demonstration; the loss is cross-entropy; it only pushes up demonstrated behavior and cannot surpass the demonstrator
- [ ] RLHF trains an RM from human preference, then optimizes the RM score, with a KL constraint against walking too far; the RM is a proxy objective and may be reward-hacked
- [ ] RLVR uses a rule-based criterion as the reward; GRPO normalizes in-group advantage and needs no value network
- [ ] All-correct and all-incorrect groups have every advantage equal to 0; those samples produce no gradient, and the compute is wasted
- [ ] PPO uses a value network as the baseline; GRPO uses the group mean as the baseline; their convergence behavior differs
- [ ] Agent post-training upgrades the training unit from text to a trajectory; the reward comes from environment execution results and may be sparse and delayed
- [ ] Trajectory-level reward has large gradient variance and slow convergence; milestone reward supplies a process signal and has small variance
- [ ] Environment-feedback flywheel: model output → environment judgment → signal as reward → filter successful trajectories and resample; failed tasks may be discarded or rewritten
- [ ] Four stages of signal source: human demonstration → human preference → rule-based criterion → environment execution result; cost falls, and hacking becomes harder


## Exercises

> You may ask an AI to explain the idea. Do not ask it to finish the exercise for you.


**Exercise 1: classify the four post-training stages**

The 6 training-setup descriptions below each belong to one stage. Complete the classify function so that each description returns "SFT" / "RLHF" / "RLVR" / "Agent RL". Reference answers are already filled in. Complete the function yourself on paper first, then run it to check.

Hint: first find the signal source in the description — demonstration, preference ranking, rule-based criterion, or environment execution result. Each description corresponds to exactly one stage.


In [ ]:
def classify(desc):
    """Judge the training stage from the signal source in the description."""
    if "trajectory" in desc or "sandbox" in desc:
        return "Agent RL"
    if "ground-truth" in desc or "rule function" in desc or "passes the tests" in desc:
        return "RLVR"
    if "demonstration" in desc or "desired reply" in desc:
        return "SFT"
    return "RLHF"


descs = [
    "Cross-entropy fine-tuning on human-written (instruction, desired reply) pairs",
    "Annotators rank 6 candidate replies, then a reward model is trained and PPO is run",
    "A rule function checks whether the answer matches the ground-truth, then GRPO is run",
    "Run a full Agent trajectory in a sandbox and use whether tests pass as the reward",
    "An AI critiques and revises itself against a list of principles, then PPO is run on preferences",
    "A rule function checks whether the code passes the tests, with reward +1/-1",
]
stages = [classify(d) for d in descs]
print("Classification:", stages)

assert stages[0] == "SFT"
assert stages[1] == "RLHF"
assert stages[2] == "RLVR"
assert stages[3] == "Agent RL"
assert stages[4] == "RLHF"
assert stages[5] == "RLVR"
print("All 6 items classified correctly. Catch the signal source, and the training stage can be located.")


**Exercise 2: in-group advantage and zero-gradient groups**

rewards is an in-group reward array. Complete grpo_advantage to return (r - mean) / std; then complete has_zero_signal, which judges whether a group of rewards produces a zero gradient (all correct or all incorrect). Reference answers are already filled in. Complete the function yourself on paper first, then run it to check.

Hint: first compute the mean, subtract the mean, then divide by the standard deviation; add a small amount (such as 1e-9) to the std denominator to prevent division by zero on all-correct/all-incorrect groups.


In [ ]:
import numpy as np


def grpo_advantage(rewards):
    """In-group normalized advantage; add a small amount to the std denominator to prevent division by zero."""
    mean = rewards.mean()
    std = rewards.std() + 1e-9
    return (rewards - mean) / std


def has_zero_signal(rewards):
    """When all are correct or all are incorrect, every advantage is 0; return True."""
    return bool(rewards.max() == rewards.min())


r2 = np.array([1.0, 1.0, -1.0, -1.0])
a2 = grpo_advantage(r2)
print("r =", r2, "-> advantage =", np.round(a2, 3))
assert np.allclose(a2, [1.0, 1.0, -1.0, -1.0])
assert has_zero_signal(np.array([1.0, 1.0, 1.0])) is True
assert has_zero_signal(np.array([-1.0, -1.0])) is True
assert has_zero_signal(r2) is False
assert np.allclose(grpo_advantage(np.array([1.0, 1.0, 1.0])), 0.0)

print("Advantage normalization and zero-gradient-group detection are both correct.")
print("All-correct/all-incorrect groups have every advantage equal to 0 and produce no gradient; a large number of such groups is wasted compute, and DAPO uses dynamic sampling to address it.")


**Exercise 3: trajectory reward and milestone reward**

A toy three-step tool-call task: query a database → filter → return a result, each step with a subgoal (whether it was reached can be judged). Complete trajectory_reward (give +1 only if all three steps succeed) and milestone_reward (+0.3 if the subgoal at that step is reached, plus +0.1 if the last step succeeds), and print how the two rewards differ on the same trajectory. Reference answers are already filled in. Complete the function yourself on paper first, then run it to check.

Hint: milestone reward is giving the model a process signal, which is MiRA's motive; first judge whether each step's subgoal was reached, then accumulate.


In [ ]:
def trajectory_reward(steps_ok):
    """steps_ok is three booleans. Give +1 only if all succeed, otherwise -1."""
    return 1.0 if all(steps_ok) else -1.0


def milestone_reward(steps_ok):
    """+0.3 if the subgoal at that step is reached, plus +0.1 if the last succeeds (total 1.0), keep one decimal place."""
    per_step = 0.3 * sum(steps_ok)
    final = 0.1 if all(steps_ok) else 0.0
    return round(per_step + final, 1)


ok_all = [True, True, True]
ok_partial = [True, False, True]

assert trajectory_reward(ok_all) == 1.0
assert trajectory_reward(ok_partial) == -1.0
assert milestone_reward(ok_all) == 1.0
assert milestone_reward(ok_partial) == 0.6

print("Trajectory reward: all correct", trajectory_reward(ok_all), ", partial", trajectory_reward(ok_partial))
print("Milestone reward: all correct", milestone_reward(ok_all), ", partial", milestone_reward(ok_partial))
print("On the same trajectory, trajectory reward gives only -1, while milestone reward gives a +0.6 process signal — the model knows the first step was correct.")


## References

- Ouyang et al., [Training language models to follow instructions with human feedback](https://arxiv.org/abs/2203.02155), 2022 — InstructGPT: the SFT→RM→PPO three-stage RLHF benchmark, the technical predecessor of ChatGPT; 1.3B matched 175B in parameter gap
- Christiano et al., [Deep Reinforcement Learning from Human Preferences](https://arxiv.org/abs/1706.03741), 2017 — the intellectual source of RLHF: learn a reward model from human preference rather than hand-write a reward
- Bai et al., [Constitutional AI: Harmlessness from AI Feedback](https://arxiv.org/abs/2212.08073), 2022 — compress human annotation cost with AI feedback; lecture 04 already read this closely
- Wei et al., [Finetuned Language Models are Zero-Shot Learners](https://arxiv.org/abs/2109.01652), 2021 — FLAN: a representative of instruction tuning, evidence from the SFT era of generalizing to new instructions
- Shao et al., [DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models](https://arxiv.org/abs/2402.03300), 2024 — where GRPO was proposed, the source of the in-group advantage idea; lecture 06 already read this closely
- Guo et al., [DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning](https://arxiv.org/abs/2501.12948), 2025 — the landmark of RLVR: R1-Zero with no SFT, pure RL, thinking emerging on its own; the source of the phrase "no RM that can be hacked"
- Yu et al., [DAPO: An Open-Source LLM Reinforcement Learning System at Scale](https://arxiv.org/abs/2503.14476), 2025 — engineering RLVR: zero gradient on all-correct/all-incorrect groups, dynamic sampling, and Clip-Higher; lecture 06 already read this closely
- Chen et al., [RLEF: Grounding Code LLMs in Execution Feedback with Reinforcement Learning](https://arxiv.org/abs/2410.02089), 2024 — execution feedback + RL teaching a model to fix code, the minimal prototype of Agent trajectory RL; lecture 04 already read this closely
- Yao et al., [WebShop: Towards Scalable Real-World Web Interaction with Grounded Language Agents](https://arxiv.org/abs/2207.01206), 2022 — an early IL+RL web-shopping Agent, the contrast starting point for Agent post-training
- Qin et al., [ToolLLM: Facilitating Large Language Models to Master 16000+ Real-world APIs](https://arxiv.org/abs/2307.16789), 2023 — an early representative of tool-call training, treating which API to call as the action
- Xu et al., [WebRL: Training LLM Web Agents via Self-Evolving Online Curriculum Reinforcement Learning](https://arxiv.org/abs/2411.02337), 2024 — self-evolving online curriculum RL for web Agents; Llama-3.1-8B on WebArena-Lite from 4.8% to 42.4%
- Wang et al., [A Subgoal-driven Framework for Improving Long-Horizon LLM Agents](https://arxiv.org/abs/2603.19685), 2026 — MiRA: milestone dense rewards for the sparse delayed reward of Agent RL; Gemma3-12B on WebArena-Lite from 6.4% to 43.0%
- OpenAI, [Advancing RL for agentic systems](https://openai.com/index/advancing-rl-for-agentic-systems/), 2025 — a frontier case of full trajectory-level RL in a remote sandbox; the reward comes from tests and terminal success or failure
